## Robomimic Get Started Tutorial

This notebook implements a simple training loop without the extensive features offered in robomimic such as logging and hyperparameter sweeping. Please refer to the [repository](https://github.com/ARISE-Initiative/robomimic) and the [documentation](https://robomimic.github.io/docs/introduction/overview.html) for the full set of features and the rest of the pipeline.

This notebook includes the following tutorials:

1. Set up robomimic development environment
2. Downloading task-specific dataset
3. Create a naive behavior cloning policy
4. Setup a simple training loop
5. Run policy training
6. Visualize the trained policy

###0. Use GPU to accelerate training

To use GPU runtime, click runtime on the top navigation part -> change runtime type -> select GPU as your accelerator

### 1. Set up development environment

The main dependencies of robomimic are
- torch
- numpy
- h5py
- robosuite
- mujoco
- tensorbordX
- egl_probe
- matplotlib


The full list is included in the requirements.txt file in the repo.

In [2]:
import os
# First, we need to decide where to host the runtime storage
USE_GDRIVE_STORAGE = False

if not USE_GDRIVE_STORAGE:
    # Option 1: use the colab runtime storage. All trained model and downloaded
    # will disappear after you disconnect from the runtime.
    WS_DIR = "/Users/a91807/Documents/GitHub/robomimic"
else:
    # Option 2: use your google drive as the runtime storage. You need to grant
    # permission for the colab runtime to access your google drive. You also
    # need to decide on a workspace for robomimic
    from google.colab import drive
    drive.mount('/content/drive')
    WS_DIR = "PATH-TO-YOUR-WORKSPACE" # this should be the absolute path, e.g., "/content/drive/MyDrive/my-ws/"
    assert os.path.exists(WS_DIR)

%cd $WS_DIR

/Users/a91807/Documents/GitHub/robomimic


In [3]:
# Clone the repo and install the basic requirements
# !git clone https://github.com/ARISE-Initiative/robomimic
# !pip install -e robomimic/

%pip install -e .
%pip install mujoco robosuite h5py imageio[ffmpeg] tensorboardX wandb

import sys
import os
# sys.path.append('./robomimic/')

Obtaining file:///Users/a91807/Documents/GitHub/robomimic
  Preparing metadata (setup.py) ... done
  Using cached egl_probe-1.0.2.tar.gz (217 kB)
  Preparing metadata (setup.py) ... done
  Using cached huggingface_hub-0.23.4-py3-none-any.whl.metadata (12 kB)
  Using cached transformers-4.41.2-py3-none-any.whl.metadata (43 kB)
  Using cached diffusers-0.11.1-py3-none-any.whl.metadata (29 kB)
  Using cached tokenizers-0.19.1-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.7 kB)
Using cached diffusers-0.11.1-py3-none-any.whl (524 kB)
Using cached huggingface_hub-0.23.4-py3-none-any.whl (402 kB)
Using cached transformers-4.41.2-py3-none-any.whl (9.1 MB)
Using cached tokenizers-0.19.1-cp312-cp312-macosx_11_0_arm64.whl (2.4 MB)
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> [361 lines of output]
      /Users/a91807/miniconda3/lib/python3.12/site-packages/setuptools/_distutils/dist.py:270: UserWarning: Unknown distributi

**WARNING**: To exactly reproduce the setup from our study paper, mujoco and robosuite should be installed from source following the [instruction](https://robomimic.github.io/docs/introduction/installation.html#install-simulators) However, if you just want to train on our dataset, you could proceed with the
following default setup.

In [4]:
# Install mujoco and robosuite
import os

# install all system dependencies for mujoco-py
# !sudo apt install curl git libgl1-mesa-dev libgl1-mesa-glx libglew-dev \
#          libosmesa6-dev software-properties-common net-tools unzip vim \
#          virtualenv wget xpra xserver-xorg-dev libglfw3-dev patchelf

#install mujoco-py
# !pip install mujoco

#install robosuite
# !pip install robosuite

In [5]:
import sys, os
PY = sys.executable
assert os.path.exists("setup.py"), "Run this from the robomimic repo root"

# Remove any partial install
!{PY} -m pip uninstall -y robomimic || true

# Editable install of robomimic itself, but SKIP dependencies (avoids egl_probe on macOS)
!{PY} -m pip install -e . --no-deps
# Core scientific/logging/video deps
!{PY} -m pip install -U numpy h5py psutil tqdm termcolor matplotlib tensorboard tensorboardX "imageio[ffmpeg]" imageio-ffmpeg

# Robotics stack
!{PY} -m pip install -U mujoco "robosuite>=1.5"

# PyTorch (Apple Silicon wheels available via PyPI nowadays)
!{PY} -m pip install -U torch torchvision torchaudio

!{PY} -m pip install -U "transformers>=4.44" "safetensors>=0.4" "accelerate>=0.25" "tokenizers>=0.15" "huggingface_hub>=0.24"

!{PY} -m pip install -U diffusers einops timm


Found existing installation: robomimic 0.5.0
Uninstalling robomimic-0.5.0:
  Successfully uninstalled robomimic-0.5.0
Obtaining file:///Users/a91807/Documents/GitHub/robomimic
  Preparing metadata (setup.py) ... done
  DEPRECATION: Legacy editable install of robomimic==0.5.0 from file:///Users/a91807/Documents/GitHub/robomimic (setup.py develop) is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or enable --use-pep517, and use setuptools >= 64. If the resulting installation is not behaving as expected, try using --config-settings editable_mode=compat. Please consult the setuptools documentation for more information. Discussion can be found at https://github.com/pypa/pip/issues/11457
  Running setup.py develop for robomimic


In [6]:
# (Optional) test robomimic installation by running a dummy training loop
# !python examples/train_bc_rnn.py --debug

## 2. Download demonstration dataset for a task

For robomimic tasks, we organize the demonstration datasets by
- task name (e.g., lift)
- data source (ph - proficient human, mh - multi human, mg - machine-generated)
- observation type (low_dim or image)

For more details of the dataset structure, visit [robomimic documentation](https://robomimic.github.io/docs/datasets/robomimic_v0.1.html) and the [dataset tutorial](https://github.com/ARISE-Initiative/robomimic/blob/master/examples/notebooks/datasets.ipynb)


Here we demonstrate downloading the proficient human (`ph`) dataset with low-dimensional (`low_dim`) observation for the `lift` task.



In [7]:
import os, sys
from glob import glob

# (1) Download the dataset to WS_DIR/datasets using the official helper
!{sys.executable} robomimic/scripts/download_datasets.py --tasks lift --dataset_types ph --output_dir "$WS_DIR/datasets"

# (2) Resolve the actual file that was downloaded (version can be v15, v141, etc.)
candidate_dir = os.path.join(WS_DIR, "datasets", "lift", "ph")
candidates = sorted(glob(os.path.join(candidate_dir, "low_dim_*.hdf5")))
assert candidates, f"No low_dim_*.hdf5 found under: {candidate_dir}"

dataset_path = candidates[-1]  # pick the newest / last match
print("Using dataset:", dataset_path)

# (3) Show dataset metadata (use the SAME interpreter as the notebook)
!{sys.executable} robomimic/scripts/get_dataset_info.py --dataset "$dataset_path"


ROBOMIMIC WARNING(
    No private macro file found!
    It is recommended to use a private macro file
    To setup, run: python /Users/a91807/Documents/GitHub/robomimic/robomimic/scripts/setup_macros.py
)
usage: download_datasets.py [-h] [--download_dir DOWNLOAD_DIR]
                            [--tasks TASKS [TASKS ...]]
                            [--dataset_types DATASET_TYPES [DATASET_TYPES ...]]
                            [--hdf5_types HDF5_TYPES [HDF5_TYPES ...]]
                            [--dry_run]
download_datasets.py: error: unrecognized arguments: --output_dir /Users/a91807/Documents/GitHub/robomimic/datasets
Using dataset: /Users/a91807/Documents/GitHub/robomimic/datasets/lift/ph/low_dim_v15.hdf5

total transitions: 9666
total trajectories: 200
traj length mean: 48.33
traj length std: 6.116461395284041
traj length min: 36
traj length max: 64
action min: -1.0
action max: 1.0

==== Filter Keys ====
filter key 20_percent with 40 demos
filter key 20_percent_train with 36 dem

In [8]:
# Here we can print out the data metadata
# !python robomimic/robomimic/scripts/get_dataset_info.py --dataset $dataset_path
# !python robomimic/scripts/get_dataset_info.py --dataset $dataset_path

In [9]:
from robomimic.scripts.extract_action_dict import extract_action_dict
extract_action_dict(dataset_path, add_absolute_actions=False)


## 3. Build a simple behavior cloning model

Follows the default hyperparameter in `robomimic/config/bc_config.py`.

In [10]:
# import all utility functions

import numpy as np

import torch
from torch.utils.data import DataLoader

import robomimic
import robomimic.utils.obs_utils as ObsUtils
import robomimic.utils.torch_utils as TorchUtils
import robomimic.utils.test_utils as TestUtils
import robomimic.utils.file_utils as FileUtils
import robomimic.utils.train_utils as TrainUtils
from robomimic.utils.dataset import SequenceDataset

from robomimic.config import config_factory
from robomimic.algo import algo_factory

In [11]:
def get_example_model(dataset_path, device):
    """
    Stronger BC: RNN (seq 20) + GMM head, gradient clipping, LR schedule.
    Action target = [rel_pos (3), rel_rot_6d (6), gripper (1)] from action_dict.
    """
    from robomimic.config import config_factory
    from robomimic.algo import algo_factory
    import robomimic.utils.obs_utils as ObsUtils
    import robomimic.utils.file_utils as FileUtils

    # --- Define the action keys ONCE and reuse in loader too ---
    ACTION_KEYS = ("action_dict/rel_pos", "action_dict/rel_rot_6d", "action_dict/gripper")  # <-- changed
    SEQ_LEN = 20

    config = config_factory(algo_name="bc")
    with config.values_unlocked():
        config.observation.modalities.obs.low_dim = [
            "robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos", "object"
        ]
        config.observation.modalities.obs.rgb = []
        config.train.hdf5_normalize_obs = True
        config.algo.rnn.enabled = True
        config.algo.rnn.horizon = SEQ_LEN
        config.algo.rnn.hidden_dim = 512
        config.algo.gaussian.enabled = False
        config.algo.gmm.enabled = True
        config.algo.gmm.num_modes = 5
        config.algo.gmm.min_std = 1e-4
        config.algo.optim_params.policy.optimizer_type = "adamw"
        config.algo.optim_params.policy.learning_rate.initial = 3e-4
        config.algo.optim_params.policy.learning_rate.decay_factor = 0.5
        config.algo.optim_params.policy.learning_rate.epoch_schedule = [120, 160, 190]
        config.algo.optim_params.policy.regularization.L2 = 1e-6
        config.train.max_grad_norm = 1.0

    ObsUtils.initialize_obs_utils_with_config(config)

    ds_cfg = {"path": dataset_path}
    shape_meta = FileUtils.get_shape_metadata_from_dataset(
        dataset_config=ds_cfg,
        all_obs_keys=tuple(config.observation.modalities.obs.low_dim),
        action_keys=ACTION_KEYS,
    )

    model = algo_factory(
        algo_name=config.algo_name,
        config=config,
        obs_key_shapes=shape_meta["all_shapes"],
        ac_dim=shape_meta["ac_dim"],
        device=device,
    )

    model._ACTION_KEYS = ACTION_KEYS
    model._SEQ_LEN = SEQ_LEN
    return model


In [12]:
device = TorchUtils.get_torch_device(try_to_use_cuda=True)
model = get_example_model(dataset_path, device=device)

print(model)


============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['robot0_eef_quat', 'object', 'robot0_gripper_qpos', 'robot0_eef_pos']
using obs modality: rgb with keys: []
using obs modality: depth with keys: []
using obs modality: scan with keys: []
ObservationKeyToModalityDict: mean not found, adding mean to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: scale not found, adding scale to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: logits not found, adding logits to mapping with assumed low_dim modality!
BC_RNN_GMM (
  ModuleDict(
    (policy): RNNGMMActorNetwork(
        action_dim=10, std_activation=softplus, low_noise_eval=True, num_nodes=5, min_std=0.0001
  
        encoder=ObservationGroupEncoder(
            group=obs
            ObservationEncoder(
                Key(
                    name=object
                    shape=[10]
                    modality=low_dim
                

## 4. Build a simple training loop

Here we build a simple data loader pipeline and a training loop. Note that this code snippet is only instructional and is a stripped-down version of robomimic's main training loop (`robomimic/scripts/train.py`).

In [14]:
def get_data_loader(dataset_path):
    ACTION_KEYS = getattr(model, "_ACTION_KEYS", ("actions",))
    SEQ_LEN = getattr(model, "_SEQ_LEN", 10)
    action_config = {k: {"normalization": None} for k in ACTION_KEYS}

    dataset = SequenceDataset(
        hdf5_path=dataset_path,
        obs_keys=("robot0_eef_pos","robot0_eef_quat","robot0_gripper_qpos","object"),
        dataset_keys=("rewards", "dones"),
        action_keys=ACTION_KEYS,
        action_config=action_config,
        load_next_obs=False,
        frame_stack=1,
        seq_length=SEQ_LEN,
        pad_frame_stack=True,
        pad_seq_length=True,
        get_pad_mask=False,
        goal_mode=None,
        hdf5_cache_mode="all",                  # "all" if you have RAM to spare
        hdf5_use_swmr=False,
        hdf5_normalize_obs=True,
        filter_by_attribute="train",                # None if file has no train mask
    )
    print("\n============= Created Dataset =============")
    print(dataset)
    print("")

    data_loader = DataLoader(
        dataset=dataset,
        sampler=None,
        batch_size=256,
        shuffle=True,
        num_workers=2,
        drop_last=True,
        pin_memory=torch.cuda.is_available(),
    )
    return data_loader


def run_train_loop(model, data_loader, num_epochs=50, gradient_steps_per_epoch=100):
    """
    Stripped-down train loop (no logging or rollouts).
    """
    model.set_train()
    for epoch in range(1, num_epochs + 1):
        data_loader_iter = iter(data_loader)
        losses = []
        for _ in range(gradient_steps_per_epoch):
            try:
                batch = next(data_loader_iter)
            except StopIteration:
                data_loader_iter = iter(data_loader)
                batch = next(data_loader_iter)

            input_batch = model.process_batch_for_training(batch)
            info = model.train_on_batch(batch=input_batch, epoch=epoch, validate=False)
            step_log = model.log_info(info)
            losses.append(step_log["Loss"])
        model.on_epoch_end(epoch)
        print(f"Train Epoch {epoch}: Loss {np.mean(losses)}")


## 5. Run policy training

Using the model and the training loop defined above. Note that this simple training loop does not save checkpoint. For model checkpointing, take a look at the full-feature [training loop](https://github.com/ARISE-Initiative/robomimic/blob/master/robomimic/scripts/train.py#L290) and the [documentation](https://robomimic.github.io/docs/tutorials/viewing_results.html)

In [40]:
# get dataset loader
data_loader = get_data_loader(dataset_path=dataset_path)

# run training loop
run_train_loop(model=model, data_loader=data_loader, num_epochs=50, gradient_steps_per_epoch=100)

SequenceDataset: normalizing observations...
100%|██████████| 179/179 [00:00<00:00, 3012.59it/s]
SequenceDataset: loading dataset into memory...
100%|██████████| 180/180 [00:00<00:00, 1905.04it/s]
SequenceDataset: caching get_item calls...
100%|██████████| 8640/8640 [00:01<00:00, 7735.46it/s]

============= Created Dataset =============
SequenceDataset (
	path=/Users/a91807/Documents/GitHub/robomimic/datasets/lift/ph/low_dim_v15.hdf5
	obs_keys=('robot0_eef_pos', 'robot0_eef_quat', 'robot0_gripper_qpos', 'object')
	seq_length=20
	filter_key=train
	frame_stack=1
	pad_seq_length=True
	pad_frame_stack=True
	goal_mode=none
	cache_mode=all
	num_demos=180
	num_sequences=8640
)

ROBOMIMIC WARNING(
    No private macro file found!
    It is recommended to use a private macro file
    To setup, run: python /Users/a91807/Documents/GitHub/robomimic/robomimic/scripts/setup_macros.py
)
ROBOMIMIC WARNING(
    No private macro file found!
    It is recommended to use a private macro file
    To setup,

## 6. Evaluate and visualize trained policy

Here we execute the trained policy `model` in a simulated environment and play the rollout video.

In [41]:
import os
os.environ["MUJOCO_GL"] = os.environ.get("MUJOCO_GL", "glfw")

from robomimic.utils import file_utils as FileUtils
from robomimic.utils import env_utils as EnvUtils

try:
    env_meta = FileUtils.get_env_metadata_from_dataset(dataset_path)
except TypeError:
    env_meta = FileUtils.get_env_metadata_from_dataset(dataset_config={"path": dataset_path})

print("env_name:", env_meta["env_name"])
# print("env_kwargs:", env_meta.get("env_kwargs", {}))

env = EnvUtils.create_env_from_metadata(
    env_meta=env_meta,
    env_name=env_meta["env_name"],
    render=True,              # set False on headless servers
    render_offscreen=False,   # set True on headless servers
    use_image_obs=False,
)

env_name: Lift
Created environment with name Lift
Action size is 7


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

from robomimic.algo import RolloutPolicy
import imageio
import robomimic.utils.torch_utils as TorchUtils

def rot6d_to_axis_angle(rot6, device=None):
    """
    rot6: np.ndarray of shape (6,) or (N,6)
    returns axis-angle np.ndarray of shape (3,) or (N,3).
    """
    if device is None:
        device = TorchUtils.get_torch_device(try_to_use_cuda=True)

    x = torch.as_tensor(rot6, dtype=torch.float32, device=device)
    if x.ndim == 1:
        x = x.unsqueeze(0)  # (1,6)

    a1 = x[:, :3]
    a2 = x[:, 3:6]

    b1 = F.normalize(a1, dim=-1)
    a2_proj = (b1 * a2).sum(dim=-1, keepdim=True) * b1
    b2 = F.normalize(a2 - a2_proj, dim=-1)
    b3 = torch.cross(b1, b2, dim=-1)

    R = torch.stack([b1, b2, b3], dim=-1)  # (N,3,3)

    # angle
    trace = R[:, 0, 0] + R[:, 1, 1] + R[:, 2, 2]
    cos = (trace - 1.0) / 2.0
    cos = torch.clamp(cos, -1.0 + 1e-7, 1.0 - 1e-7)
    angle = torch.acos(cos)  # (N,)

    # axis
    sin = torch.sin(angle)
    axis = torch.zeros_like(b1)
    mask = sin.abs() > 1e-4
    if mask.any():
        num = torch.stack([
            R[:, 2, 1] - R[:, 1, 2],
            R[:, 0, 2] - R[:, 2, 0],
            R[:, 1, 0] - R[:, 0, 1],
        ], dim=-1)  # (N,3)
        axis[mask] = F.normalize(num[mask] / (2.0 * sin[mask].unsqueeze(-1)), dim=-1)
    if (~mask).any():
        axis[~mask] = torch.tensor([1.0, 0.0, 0.0], device=device)

    axis_angle = axis * angle.unsqueeze(-1)
    aa_np = axis_angle.detach().cpu().numpy()
    if aa_np.shape[0] == 1:
        return aa_np[0]
    return aa_np

def rollout_10d_to_7d(policy_10d, env, horizon=200, video_path="rollout.mp4", device=None):
    """
    policy_10d: RolloutPolicy(model) that outputs 10-D actions:
                [pos(3), rot6(6), gripper(1)]
    env: robosuite env expecting 7-D actions:
         [pos(3), axis_angle(3), gripper(1)]
    """
    if device is None:
        device = TorchUtils.get_torch_device(try_to_use_cuda=True)

    # video writer
    writer = imageio.get_writer(video_path, fps=20)

    # reset
    if hasattr(policy_10d, "reset"):
        policy_10d.reset()
    obs = env.reset()
    total_r = 0.0
    last_info = {}

    for t in range(horizon):
        ac10 = policy_10d(ob=obs)
        ac10 = np.asarray(ac10, dtype=np.float32)
        assert ac10.shape[-1] == 10, f"Expected 10-D action, got {ac10.shape}"

        pos = ac10[:3]
        rot6 = ac10[3:9]
        grip = ac10[9:]

        aa = rot6d_to_axis_angle(rot6, device=device)  # (3,)
        ac7 = np.concatenate([pos, aa, grip], axis=-1) # (7,)

        # step env
        obs, r, done, info = env.step(ac7)
        total_r += float(r)
        last_info = info

        # write frame
        frame = env.render(mode="rgb_array", height=512, width=512, camera_name="agentview")
        writer.append_data(frame)

        if done:
            break

    writer.close()
    return {"Return": total_r, "Horizon": t + 1, "info": last_info}

from robomimic.algo import RolloutPolicy

device = TorchUtils.get_torch_device(try_to_use_cuda=True)

model.nets.eval()

policy_10d = RolloutPolicy(model)

rollout_log = rollout_10d_to_7d(policy_10d, env, horizon=200, video_path="rollout.mp4", device=device)
print(rollout_log)


{'Return': 159.0, 'Horizon': 200, 'info': {'is_success': {'task': np.True_}}}


In [49]:
# visualize rollout video

from IPython.display import HTML
from base64 import b64encode

video_path = "rollout.mp4"

mp4 = open(video_path, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f"""
<video width=400 controls>
      <source src="{data_url}" type="video/mp4">
</video>
""")

## 7. Preference pairs + DPO



In [ ]:
# Preference dataset collection (BC as reference & collector)
import os, pickle
from dataclasses import dataclass
from typing import List, Tuple, Optional
import numpy as np
import torch
import torch.nn.functional as F

import robomimic.utils.torch_utils as TorchUtils
from robomimic.algo import RolloutPolicy

# MuJoCo state helpers
def _base_env(e):
    cur = e
    for name in ("unwrapped", "env"):
        if hasattr(cur, name):
            cur = getattr(cur, name)
    return cur

def save_mj_state(env):
    base = _base_env(env)
    sim = base.sim
    m, d = sim.model, sim.data
    st = {
        "qpos": d.qpos.copy(),
        "qvel": d.qvel.copy(),
        "act": d.act.copy() if getattr(m, "na", 0) > 0 else None,
        "mocap_pos": d.mocap_pos.copy() if getattr(m, "nmocap", 0) > 0 else None,
        "mocap_quat": d.mocap_quat.copy() if getattr(m, "nmocap", 0) > 0 else None,
        "time": float(d.time),
    }
    return st

def load_mj_state(env, st):
    base = _base_env(env)
    sim = base.sim
    m, d = sim.model, sim.data
    d.qpos[...] = st["qpos"]
    d.qvel[...] = st["qvel"]
    if st.get("act") is not None and getattr(m, "na", 0) > 0:
        d.act[...] = st["act"]
    if st.get("mocap_pos") is not None and getattr(m, "nmocap", 0) > 0:
        d.mocap_pos[...] = st["mocap_pos"]
        d.mocap_quat[...] = st["mocap_quat"]
    if "time" in st:
        d.time = float(st["time"])
    if hasattr(sim, "forward"):
        sim.forward()
    else:
        try:
            from robosuite.utils.binding_utils import mj_forward as _mj_forward
            _mj_forward(m, d)
        except Exception:
            pass

def start_hash(st) -> bytes:
    return st["qpos"].tobytes() + st["qvel"].tobytes()

# Success helper
def extract_success(info: dict) -> bool:
    """
    Infer task success from env info. Primary key in your env is:
      info["is_success"]["task"] (np.bool_).
    Fallbacks provided for robustness.
    """
    if not isinstance(info, dict):
        return False

    if "is_success" in info:
        succ = info["is_success"]
        # common pattern: {"task": np.bool_}
        if isinstance(succ, dict) and "task" in succ:
            return bool(succ["task"])
        try:
            return bool(succ)
        except Exception:
            pass

    for k in ("success", "task_success", "goal_achieved"):
        if k in info:
            try:
                return bool(info[k])
            except Exception:
                pass
    return False

# Obs & Trajectory
@dataclass
class Trajectory:
    obs_flat: List[np.ndarray]
    obs_dicts: List[dict]
    actions: List[np.ndarray]      # keep original 10-D actions here
    ret: float
    start_sig: bytes
    success: bool 
    bc_logps: Optional[List[float]] = None

DEFAULT_OBS_KEYS = ("robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos", "object")

def pick_flatten_keys(obs0: dict) -> Tuple[str, ...]:
    preferred = [k for k in DEFAULT_OBS_KEYS if k in obs0]
    if preferred:
        return tuple(preferred)
    keys = []
    for k, v in obs0.items():
        a = np.asarray(v)
        if a.dtype != np.bool_ and np.issubdtype(a.dtype, np.number):
            keys.append(k)
    return tuple(sorted(keys))

def flatten_obs(obs: dict, keys: Tuple[str, ...]) -> np.ndarray:
    parts = []
    for k in keys:
        a = np.asarray(obs[k], dtype=np.float32).ravel()
        parts.append(a)
    return np.concatenate(parts, axis=0).astype(np.float32)

# 6D rotation -> axis-angle for action conversion
def rot6d_to_axis_angle(rot6: np.ndarray, device=None) -> np.ndarray:
    """
    rot6: shape (6,)
    returns axis-angle: shape (3,)
    """
    if device is None:
        device = TorchUtils.get_torch_device(try_to_use_cuda=True)

    x = torch.as_tensor(rot6, dtype=torch.float32, device=device).view(1, 6)
    a1 = x[:, :3]
    a2 = x[:, 3:6]

    b1 = F.normalize(a1, dim=-1)
    a2_proj = (b1 * a2).sum(dim=-1, keepdim=True) * b1
    b2 = F.normalize(a2 - a2_proj, dim=-1)
    b3 = torch.cross(b1, b2, dim=-1)

    R = torch.stack([b1, b2, b3], dim=-1)  # (1,3,3)

    trace = R[:, 0, 0] + R[:, 1, 1] + R[:, 2, 2]
    cos = (trace - 1.0) / 2.0
    cos = torch.clamp(cos, -1.0 + 1e-7, 1.0 - 1e-7)
    angle = torch.acos(cos)  # (1,)

    sin = torch.sin(angle)
    axis = torch.zeros_like(b1)
    mask = sin.abs() > 1e-4
    if mask.any():
        num = torch.stack([
            R[:, 2, 1] - R[:, 1, 2],
            R[:, 0, 2] - R[:, 2, 0],
            R[:, 1, 0] - R[:, 0, 1],
        ], dim=-1)
        axis[mask] = F.normalize(num[mask] / (2.0 * sin[mask].unsqueeze(-1)), dim=-1)
    if (~mask).any():
        axis[~mask] = torch.tensor([1.0, 0.0, 0.0], device=device)

    axis_angle = axis * angle.unsqueeze(-1)
    return axis_angle.squeeze(0).detach().cpu().numpy()

def action_10d_to_7d(a10: np.ndarray, device=None) -> np.ndarray:
    """
    a10: [pos(3), rot6(6), grip(1)] -> [pos(3), axis_angle(3), grip(1)]
    """
    a10 = np.asarray(a10, dtype=np.float32)
    assert a10.shape[-1] == 10, f"Expected 10-D action, got {a10.shape}"
    pos = a10[:3]
    rot6 = a10[3:9]
    grip = a10[9:]
    aa = rot6d_to_axis_angle(rot6, device=device)
    return np.concatenate([pos, aa, grip], axis=-1)

# BC action
def bc_act(policy_bc, obs: dict, deterministic: bool = False):
    def parse_out(out):
        if isinstance(out, dict):
            a = None
            for k in ("actions", "action", "raw_actions", "raw_action"):
                if k in out:
                    a = out[k]; break
            if a is None:
                raise RuntimeError("BC dict without action key")
            lp = out.get("logp", out.get("log_prob", out.get("log_pi", None)))
            a = np.asarray(a, dtype=np.float32).reshape(-1)
            lp = float(lp) if lp is not None else None
            return a, lp
        if isinstance(out, (tuple, list)) and len(out) >= 1:
            a = np.asarray(out[0], dtype=np.float32).reshape(-1)
            lp = None
            if len(out) > 1 and isinstance(out[1], dict):
                info = out[1]
                lp = info.get("logp", info.get("log_prob", info.get("log_pi", None)))
                lp = float(lp) if lp is not None else None
            return a, lp
        return np.asarray(out, dtype=np.float32).reshape(-1), None

    attempted = []
    try:
        out = policy_bc(obs)
        return parse_out(out)
    except Exception as e:
        attempted.append(("__call__", {}, repr(e)))

    for kwargs in (
        {"deterministic": deterministic, "return_log_prob": True},
        {"deterministic": deterministic, "return_log_probs": True},
        {"deterministic": deterministic},
        {"greedy": deterministic},
        {},
    ):
        try:
            out = policy_bc(obs, **kwargs)
            return parse_out(out)
        except Exception as e:
            attempted.append(("__call__", kwargs, repr(e)))

    for name in ("get_action", "act"):
        if hasattr(policy_bc, name):
            fn = getattr(policy_bc, name)
            for kwargs in (
                {"deterministic": deterministic, "return_log_prob": True},
                {"deterministic": deterministic, "return_log_probs": True},
                {"deterministic": deterministic},
                {"greedy": deterministic},
                {},
            ):
                try:
                    out = fn(obs, **kwargs)
                    return parse_out(out)
                except Exception as e:
                    attempted.append((name, kwargs, repr(e)))

    for attr in ("policy", "algo", "model"):
        sub = getattr(policy_bc, attr, None)
        if sub is None:
            continue
        for name in ("get_action", "act", "__call__"):
            fn = getattr(sub, name, None)
            if fn is None:
                continue
            for kwargs in ({}, {"deterministic": deterministic}, {"greedy": deterministic}):
                try:
                    out = fn(obs, **kwargs) if name != "__call__" else sub(obs, **kwargs)
                    return parse_out(out)
                except Exception:
                    pass

    raise AttributeError(f"bc_act could not query action; attempts (first few): {attempted[:5]}")

# Rollout and pair collection
def rollout_from_saved_start(env, policy_bc, start_state, obs0, flat_keys,
                             horizon=200, det=False) -> Trajectory:
    """
    Roll out from a saved MuJoCo state, using BC policy.
    Records obs, actions, return, and a success flag based on env info.
    """
    _ = env.reset()
    load_mj_state(env, start_state)
    if hasattr(policy_bc, "reset"):
        try:
            policy_bc.reset()
        except Exception:
            pass

    obs = obs0
    obs_flat_seq, obs_dict_seq, act_seq, bc_lps = [], [], [], []
    total_r = 0.0
    device = TorchUtils.get_torch_device(try_to_use_cuda=True)
    last_info = {}

    for _t in range(horizon):
        a10, lp = bc_act(policy_bc, obs, deterministic=det)   # 10-D from BC
        a7 = action_10d_to_7d(a10, device=device)            # 7-D for env

        next_obs, r, done, info = env.step(a7)
        obs_flat_seq.append(flatten_obs(obs, flat_keys))
        obs_dict_seq.append(obs)
        act_seq.append(a10)                                  # store original 10-D
        if lp is not None:
            bc_lps.append(lp)
        total_r += float(r)
        obs = next_obs
        last_info = info
        if done:
            break

    success_flag = extract_success(last_info)
    bc_logps = bc_lps if len(bc_lps) == len(act_seq) else None

    return Trajectory(
        obs_flat=obs_flat_seq,
        obs_dicts=obs_dict_seq,
        actions=act_seq,
        ret=total_r,
        start_sig=start_hash(start_state),
        success=success_flag,
        bc_logps=bc_logps,
    )

def collect_preference_pairs_from_bc(env,
                                     policy_bc,
                                     n_pairs=200,
                                     horizon=200,
                                     trials_per_start=60,
                                     min_return_gap=1e-6,
                                     prefer_higher=True,
                                     max_pairs_per_start=10,
                                     verbose_every=10):
    """
    Collect preference pairs from a BC policy by:
      - Sampling a start state.
      - Running `trials_per_start` rollouts from that state.
      - Splitting rollouts into successes and failures.
      - Creating pairs with preference:
          success > failure  (strong)
          success > success  (higher return)
          failure > failure  (higher return; fallback)
    """
    pairs, seen = [], set()

    while len(pairs) < n_pairs:
        obs0 = env.reset()
        start_state = save_mj_state(env)
        sig = start_hash(start_state)
        if sig in seen:
            continue
        flat_keys = pick_flatten_keys(obs0)

        trajs = []
        for _ in range(trials_per_start):
            tr = rollout_from_saved_start(
                env, policy_bc,
                start_state=start_state,
                obs0=obs0,
                flat_keys=flat_keys,
                horizon=horizon,
                det=False,
            )
            trajs.append(tr)

        if len(trajs) < 2:
            continue

        successes = [tr for tr in trajs if tr.success]
        failures  = [tr for tr in trajs if not tr.success]

        local_pairs = []

        # success vs failure
        if successes and failures:
            succ_sorted = sorted(successes, key=lambda t: t.ret, reverse=True)
            fail_sorted = sorted(failures,  key=lambda t: t.ret, reverse=True)
            for s in succ_sorted:
                for f in fail_sorted:
                    if abs(s.ret - f.ret) >= min_return_gap:
                        local_pairs.append((s, f))
                    if len(local_pairs) >= max_pairs_per_start:
                        break
                if len(local_pairs) >= max_pairs_per_start:
                    break

        # success vs success
        if len(successes) >= 2:
            succ_sorted = sorted(successes, key=lambda t: t.ret, reverse=True)
            for i in range(len(succ_sorted) - 1):
                pos, neg = succ_sorted[i], succ_sorted[i + 1]
                if abs(pos.ret - neg.ret) >= min_return_gap:
                    local_pairs.append((pos, neg))
                    if len(local_pairs) >= max_pairs_per_start:
                        break

        # failure vs failure
        if not local_pairs and len(failures) >= 2:
            fail_sorted = sorted(failures, key=lambda t: t.ret, reverse=True)
            for i in range(len(fail_sorted) - 1):
                pos, neg = fail_sorted[i], fail_sorted[i + 1]
                if abs(pos.ret - neg.ret) >= min_return_gap:
                    local_pairs.append((pos, neg))
                    if len(local_pairs) >= max_pairs_per_start:
                        break

        if not local_pairs:
            # Nothing useful from this start state
            seen.add(sig)
            continue

        np.random.shuffle(local_pairs)
        for pos, neg in local_pairs:
            if len(pairs) >= n_pairs:
                break
            if prefer_higher and pos.ret < neg.ret:
                pos, neg = neg, pos
            pairs.append((pos, neg))

        seen.add(sig)

        if verbose_every and (len(pairs) % verbose_every == 0):
            last_pos, last_neg = pairs[-1]
            print(f"[collect] {len(pairs)}/{n_pairs} pairs — "
                  f"pos_ret={last_pos.ret:.1f} (succ={last_pos.success})  "
                  f"neg_ret={last_neg.ret:.1f} (succ={last_neg.success})")

    return pairs

# Run
policy_bc = RolloutPolicy(model)

N_PAIRS     = 200
HORIZON     = 200
TRIALS_PER  = 60

print("Collecting preference pairs with BC (stochastic), grouped by start state...")
pairs = collect_preference_pairs_from_bc(
    env,
    policy_bc,
    n_pairs=N_PAIRS,
    horizon=HORIZON,
    trials_per_start=TRIALS_PER,
    min_return_gap=1e-6,
    prefer_higher=True,
    max_pairs_per_start=10,
    verbose_every=max(1, N_PAIRS // 10),
)

print(f"Collected {len(pairs)} pairs.")
os.makedirs("prefs_out", exist_ok=True)
with open("prefs_out/pairs.pkl", "wb") as f:
    pickle.dump(pairs, f)
print("Saved preference pairs to prefs_out/pairs.pkl")


[collect] 200/200 pairs — pos_ret=159.0 (succ=True)  neg_ret=140.0 (succ=False)
Collected 200 pairs.
Saved preference pairs to prefs_out/pairs.pkl


In [ ]:
# 2 rollouts per start, 1 pair
import os
import pickle
import numpy as np
from robomimic.algo import RolloutPolicy

def collect_preference_pairs_minimal(
    env,
    policy,
    n_pairs=200,
    horizon=200,
    min_return_gap=1e-6,
    prefer_higher=True,
    verbose_every=10,
):
    """
    Minimal preference dataset collector:
      - For each new start state:
          * run exactly TWO rollouts from that state with the given policy
          * build ONE (pos, neg) pair from those two trajectories
      - pos / neg logic:
          * if exactly one trajectory is successful -> success is pos
          * else -> higher-return trajectory is pos (if prefer_higher=True)
      - skip a start if the two trajs are too similar (same success & |Δret| < min_return_gap)
    """
    pairs = []
    seen = set()

    while len(pairs) < n_pairs:
        # 1) Sample a fresh start state
        obs0 = env.reset()
        start_state = save_mj_state(env)
        sig = start_hash(start_state)
        if sig in seen:
            continue
        flat_keys = pick_flatten_keys(obs0)

        # 2) Roll out exactly two trajectories from this state
        trajs = []
        for _ in range(2):
            tr = rollout_from_saved_start(
                env,
                policy,
                start_state=start_state,
                obs0=obs0,
                flat_keys=flat_keys,
                horizon=horizon,
                det=False,
            )
            trajs.append(tr)

        if len(trajs) < 2:
            seen.add(sig)
            continue

        t1, t2 = trajs

        # 3) Decide pos / neg
        if t1.success and not t2.success:
            pos, neg = t1, t2
        elif t2.success and not t1.success:
            pos, neg = t2, t1
        else:
            # both success or both failure -> use return
            if prefer_higher and t1.ret < t2.ret:
                pos, neg = t2, t1
            else:
                pos, neg = t1, t2

        # 4) Optional: skip if too similar (same success & almost same return)
        if (pos.success == neg.success) and (abs(pos.ret - neg.ret) < min_return_gap):
            seen.add(sig)
            continue

        pairs.append((pos, neg))
        seen.add(sig)

        if verbose_every and (len(pairs) % verbose_every == 0):
            print(
                f"[collect-min] {len(pairs)}/{n_pairs} pairs — "
                f"pos_ret={pos.ret:.1f} (succ={pos.success})  "
                f"neg_ret={neg.ret:.1f} (succ={neg.success})"
            )

    return pairs

policy_bc = RolloutPolicy(model)

N_PAIRS  = 200
HORIZON  = 200

print("Collecting preference pairs with minimal collector (2 rollouts per start)...")
pairs = collect_preference_pairs_minimal(
    env,
    policy_bc,
    n_pairs=N_PAIRS,
    horizon=HORIZON,
    min_return_gap=1e-6,
    prefer_higher=True,
    verbose_every=max(1, N_PAIRS // 10),
)

print(f"Collected {len(pairs)} pairs.")
os.makedirs("prefs_out", exist_ok=True)
with open("prefs_out/pairs.pkl", "wb") as f:
    pickle.dump(pairs, f)
print("Saved preference pairs to prefs_out/pairs.pkl")


[collect-min] 20/200 pairs — pos_ret=158.0 (succ=True)  neg_ret=0.0 (succ=False)
[collect-min] 40/200 pairs — pos_ret=160.0 (succ=True)  neg_ret=0.0 (succ=False)
[collect-min] 60/200 pairs — pos_ret=149.0 (succ=False)  neg_ret=140.0 (succ=False)
[collect-min] 80/200 pairs — pos_ret=118.0 (succ=False)  neg_ret=0.0 (succ=False)
[collect-min] 100/200 pairs — pos_ret=125.0 (succ=False)  neg_ret=0.0 (succ=False)
[collect-min] 120/200 pairs — pos_ret=156.0 (succ=True)  neg_ret=0.0 (succ=False)
[collect-min] 140/200 pairs — pos_ret=145.0 (succ=False)  neg_ret=0.0 (succ=False)
[collect-min] 160/200 pairs — pos_ret=151.0 (succ=False)  neg_ret=141.0 (succ=False)
[collect-min] 180/200 pairs — pos_ret=158.0 (succ=True)  neg_ret=0.0 (succ=False)
[collect-min] 200/200 pairs — pos_ret=39.0 (succ=True)  neg_ret=0.0 (succ=False)
Collected 200 pairs.
Saved preference pairs to prefs_out/pairs.pkl


In [ ]:
# Training
import os, pickle, copy
import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim

import robomimic.utils.torch_utils as TorchUtils
from robomimic.algo import RolloutPolicy

device = TorchUtils.get_torch_device(try_to_use_cuda=True)

# Helper: move all nets in a robomimic algo to device
def algo_to_device(algo, device):
    """
    algo: robomimic Algo subclass (e.g., BC_RNN_GMM)
    Moves its internal nets to the given device.
    """
    for net in getattr(algo, "nets", {}).values():
        net.to(device)
    return algo

def traj_obs_to_torch(tr, device):
    """
    Build batched obs_dict for a single trajectory:
      tr.obs_dicts: list of obs_t (dicts) over time
    Returns:
      obs_torch: dict k -> tensor [1, T, ...]
      A_torch: tensor [1, T, act_dim] from tr.actions
    """
    assert len(tr.obs_dicts) == len(tr.actions)
    T = len(tr.obs_dicts)
    keys = tr.obs_dicts[0].keys()

    obs_torch = {}
    for k in keys:
        arr = np.stack([np.asarray(obs[k], dtype=np.float32) for obs in tr.obs_dicts], axis=0)  # [T, ...]
        obs_torch[k] = torch.as_tensor(arr, device=device).unsqueeze(0)  # [1, T, ...]

    A = np.stack(tr.actions, axis=0).astype(np.float32)   # [T, act_dim]
    A_torch = torch.as_tensor(A, device=device).unsqueeze(0)  # [1, T, act_dim]

    return obs_torch, A_torch

# θ: trainable BC policy, log-prob (per-step average)
class BCTrajectoryThetaTorch:
    """
    θ_policy: BC algo instance (same architecture as your trained BC)
    Define log pθ(traj) as the average per-step log-likelihood under the RNN-GMM policy.
    """
    def __init__(self, bc_algo, device=None):
        self.model = bc_algo
        self._device = device or TorchUtils.get_torch_device(try_to_use_cuda=True)

    @property
    def device(self):
        return self._device

    def logprob_traj_tensor(self, tr) -> torch.Tensor:
        """
        True trajectory log-likelihood under θ's RNN-GMM policy:
          log pθ(τ) = (1/T) * sum_t log pθ(a_t | s_1:t).
        """
        import torch.distributions as td

        obs_torch, A_torch = traj_obs_to_torch(tr, self.device)  # [1, T, ...], [1, T, act_dim]
        policy_net = self.model.nets["policy"]

        # Single forward over the whole sequence
        out = policy_net.forward_train(
            obs_dict=obs_torch,
            goal_dict=None,
            rnn_init_state=None,
            return_state=False,
        )

        # Get the distribution object
        if isinstance(out, td.Distribution):
            dist = out
        elif isinstance(out, tuple):
            actions = out[0]
            dist_info = out[1] if len(out) > 1 else None
            if isinstance(dist_info, td.Distribution):
                dist = dist_info
            elif isinstance(dist_info, dict) and "dist" in dist_info and isinstance(dist_info["dist"], td.Distribution):
                dist = dist_info["dist"]
            else:
                raise RuntimeError("forward_train tuple does not contain a Distribution; please adapt.")
        elif isinstance(out, dict):
            if "dist" in out and isinstance(out["dist"], td.Distribution):
                dist = out["dist"]
            else:
                raise RuntimeError("forward_train dict output has no 'dist' Distribution; please adapt.")
        else:
            raise RuntimeError("Unsupported output type from forward_train for θ; need a Distribution.")

        # Compute per-step log-probs and mean over time
        logp = dist.log_prob(A_torch)    # shape [1, T] or [1, T, 1]
        logp = logp.mean()               # scalar = average per-step log-likelihood
        return logp

class BCTrajectoryReference:
    """
    Frozen reference BC policy, using the same per-step avg log-likelihood as θ.
    """
    def __init__(self, bc_algo, device=None):
        self.model = bc_algo
        self.device = device or TorchUtils.get_torch_device(try_to_use_cuda=True)

    @torch.no_grad()
    def logprob_traj_tensor(self, tr) -> torch.Tensor:
        """
        Trajectory log-likelihood under the frozen BC policy (per-step average).
        """
        import torch.distributions as td

        obs_torch, A_torch = traj_obs_to_torch(tr, self.device)
        policy_net = self.model.nets["policy"]

        out = policy_net.forward_train(
            obs_dict=obs_torch,
            goal_dict=None,
            rnn_init_state=None,
            return_state=False,
        )

        if isinstance(out, td.Distribution):
            dist = out
        elif isinstance(out, tuple):
            actions = out[0]
            dist_info = out[1] if len(out) > 1 else None
            if isinstance(dist_info, td.Distribution):
                dist = dist_info
            elif isinstance(dist_info, dict) and "dist" in dist_info and isinstance(dist_info["dist"], td.Distribution):
                dist = dist_info["dist"]
            else:
                raise RuntimeError("forward_train tuple does not contain a Distribution for ref; please adapt.")
        elif isinstance(out, dict):
            if "dist" in out and isinstance(out["dist"], td.Distribution):
                dist = out["dist"]
            else:
                raise RuntimeError("forward_train dict output has no 'dist' Distribution for ref.")
        else:
            raise RuntimeError("Unsupported output type from forward_train for ref; need a Distribution.")

        logp = dist.log_prob(A_torch)   # [1, T] or [1, T, 1]
        logp = logp.mean()              # average per-step log-likelihood
        return logp

# DPO losses
def dpo_loss(pi_logps, ref_logps, yw_idxs, yl_idxs, beta: float):
    """
    pi_logps: trajectory logprobs under θ, shape (B,)
    ref_logps: trajectory logprobs under reference, shape (B,)
    yw_idxs: preferred indices in [0, B-1], shape (T,)
    yl_idxs: dispreferred indices in [0, B-1], shape (T,)
    beta: temperature controlling KL strength
    """
    pi_yw_logps  = pi_logps[yw_idxs]
    pi_yl_logps  = pi_logps[yl_idxs]
    ref_yw_logps = ref_logps[yw_idxs]
    ref_yl_logps = ref_logps[yl_idxs]

    pi_logratios  = pi_yw_logps  - pi_yl_logps
    ref_logratios = ref_yw_logps - ref_yl_logps

    logits = beta * (pi_logratios - ref_logratios)
    # clamp logits to prevent huge magnitudes
    logits = torch.clamp(logits, min=-20.0, max=20.0)

    losses  = -F.logsigmoid(logits)
    rewards = beta * (pi_logps - ref_logps).detach()
    return losses, rewards

@torch.no_grad()
def dpo_full_loss(policy_theta, ref, pairs, beta=5e-7):
    """
    Full-dataset DPO loss for logging.
    """
    device = policy_theta.device
    pi_list, ref_list = [], []
    yw_idxs, yl_idxs = [], []

    for i, (pos, neg) in enumerate(pairs):
        idx_pos = 2 * i
        idx_neg = 2 * i + 1

        pi_list.append(policy_theta.logprob_traj_tensor(pos))
        pi_list.append(policy_theta.logprob_traj_tensor(neg))
        ref_list.append(ref.logprob_traj_tensor(pos))
        ref_list.append(ref.logprob_traj_tensor(neg))

        yw_idxs.append(idx_pos)
        yl_idxs.append(idx_neg)

    if not pi_list:
        return 0.0

    pi_logps  = torch.stack(pi_list).to(device)   # (2 * N_pairs,)
    ref_logps = torch.stack(ref_list).to(device)

    yw = torch.as_tensor(yw_idxs, device=device, dtype=torch.long)
    yl = torch.as_tensor(yl_idxs, device=device, dtype=torch.long)

    losses, _ = dpo_loss(pi_logps, ref_logps, yw, yl, beta)
    return float(losses.mean().item())

def dpo_step(policy_theta, ref, batch_pairs, beta=2.0, optimizer=None, bc_anchor_lambda: float = 0.0):
    """
    One DPO update step using trajectory log-likelihood and dpo_loss.
    """
    assert optimizer is not None
    device = policy_theta.device

    for net in policy_theta.model.nets.values():
        net.train()

    pi_list, ref_list = [], []
    yw_idxs, yl_idxs = [], []

    for i, (pos, neg) in enumerate(batch_pairs):
        idx_pos = 2 * i
        idx_neg = 2 * i + 1

        pi_list.append(policy_theta.logprob_traj_tensor(pos))
        pi_list.append(policy_theta.logprob_traj_tensor(neg))
        with torch.no_grad():
            ref_list.append(ref.logprob_traj_tensor(pos))
            ref_list.append(ref.logprob_traj_tensor(neg))

        yw_idxs.append(idx_pos)
        yl_idxs.append(idx_neg)

    pi_logps  = torch.stack(pi_list).to(device)
    ref_logps = torch.stack(ref_list).to(device)

    yw = torch.as_tensor(yw_idxs, device=device, dtype=torch.long)
    yl = torch.as_tensor(yl_idxs, device=device, dtype=torch.long)

    losses, rewards = dpo_loss(pi_logps, ref_logps, yw, yl, beta)
    loss = losses.mean()

    # Optional BC anchor
    if bc_anchor_lambda > 0.0:
        bc_penalty = torch.zeros_like(loss)
        loss = loss + bc_anchor_lambda * bc_penalty

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return float(loss.item())

def train_dpo(policy_theta, ref, pairs, beta=5e-7, lr=1e-5, steps=50,
              batch_size=32, print_every=1, full_eval_every=10, bc_anchor_lambda: float = 0.0):
    """
    DPO training loop using dpo_loss over per-step avg trajectory log-likelihoods.
    """
    params = []
    for net in policy_theta.model.nets.values():
        params += list(net.parameters())
    opt = torch.optim.Adam(params, lr=lr)

    batch_losses, full_losses = [], []
    for t in range(steps):
        idx = np.random.choice(len(pairs), size=min(batch_size, len(pairs)), replace=False)
        batch = [pairs[i] for i in idx]
        b = dpo_step(policy_theta, ref, batch, beta=beta, optimizer=opt, bc_anchor_lambda=bc_anchor_lambda)
        batch_losses.append(b)

        full = None
        if full_eval_every and (t+1) % full_eval_every == 0:
            full = dpo_full_loss(policy_theta, ref, pairs, beta=beta)
            full_losses.append((t+1, full))
        if print_every and (t+1) % print_every == 0:
            msg = f"[DPO] step {t+1:03d}/{steps}  batch_loss={b:.4f}"
            if full is not None:
                msg += f"  full_loss={full:.4f}"
            print(msg)
    return {"batch": batch_losses, "full": full_losses}

@torch.no_grad()
def pref_accuracy(policy_theta, ref, pairs, beta=5e-7):
    """
    Fraction of pairs where Δ > 0 using the same log-likelihood structure.
    """
    device = policy_theta.device
    pi_list, ref_list = [], []
    yw_idxs, yl_idxs = [], []

    for i, (pos, neg) in enumerate(pairs):
        idx_pos = 2 * i
        idx_neg = 2 * i + 1

        pi_list.append(policy_theta.logprob_traj_tensor(pos))
        pi_list.append(policy_theta.logprob_traj_tensor(neg))
        ref_list.append(ref.logprob_traj_tensor(pos))
        ref_list.append(ref.logprob_traj_tensor(neg))

        yw_idxs.append(idx_pos)
        yl_idxs.append(idx_neg)

    if not pi_list:
        return 0.0

    pi_logps  = torch.stack(pi_list).to(device)
    ref_logps = torch.stack(ref_list).to(device)
    yw = torch.as_tensor(yw_idxs, device=device, dtype=torch.long)
    yl = torch.as_tensor(yl_idxs, device=device, dtype=torch.long)

    pi_yw  = pi_logps[yw];  pi_yl  = pi_logps[yl]
    ref_yw = ref_logps[yw]; ref_yl = ref_logps[yl]
    Delta  = (pi_yw - ref_yw) - (pi_yl - ref_yl)

    return float((Delta > 0).float().mean().item())

# Train
if "pairs" not in globals():
    with open("prefs_out/pairs.pkl", "rb") as f:
        pairs_sf = pickle.load(f)

bc_model = algo_to_device(model, device)   # trained BC algo on device

# reference BC (frozen) as algo
ref = BCTrajectoryReference(bc_model, device=device)

# θ: trainable BC copy
theta_model = get_example_model(dataset_path, device=device)
for name, net in theta_model.nets.items():
    if name in bc_model.nets:
        net.load_state_dict(bc_model.nets[name].state_dict())

policy_theta = BCTrajectoryThetaTorch(theta_model, device=device)

print("Pref-accuracy before DPO:",
      pref_accuracy(policy_theta, ref, pairs_sf, beta=5e-7))

hist = train_dpo(
    policy_theta,
    ref,
    pairs_sf,        # use only success-vs-failure pairs
    beta=5e-7,
    lr=1e-6,         # smaller LR
    steps=200,        # fewer steps
    batch_size=32,
    print_every=1,
    full_eval_every=5,
    bc_anchor_lambda=0.0,
)

print("Pref-accuracy after gentle DPO:",
      pref_accuracy(policy_theta, ref, pairs_sf, beta=5e-7))

os.makedirs("prefs_out", exist_ok=True)

theta_state = {name: net.state_dict() for name, net in theta_model.nets.items()}
torch.save(theta_state, "prefs_out/bc_theta_dpo_nets.pt")

print("Saved θ to prefs_out/bc_theta_dpo_nets.pt")

policy_dpo_model = theta_model



============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['robot0_eef_quat', 'object', 'robot0_gripper_qpos', 'robot0_eef_pos']
using obs modality: rgb with keys: []
using obs modality: depth with keys: []
using obs modality: scan with keys: []
Pref-accuracy before DPO: 0.12173912674188614
[DPO] step 001/200  batch_loss=4.4422
[DPO] step 002/200  batch_loss=6.6875
[DPO] step 003/200  batch_loss=5.6415
[DPO] step 004/200  batch_loss=4.2347
[DPO] step 005/200  batch_loss=4.7462  full_loss=5.4969
[DPO] step 006/200  batch_loss=5.4232
[DPO] step 007/200  batch_loss=3.9968
[DPO] step 008/200  batch_loss=4.7705
[DPO] step 009/200  batch_loss=5.8196
[DPO] step 010/200  batch_loss=4.9113  full_loss=5.4627
[DPO] step 011/200  batch_loss=3.8603
[DPO] step 012/200  batch_loss=3.8627
[DPO] step 013/200  batch_loss=5.7072
[DPO] step 014/200  batch_loss=5.2838
[DPO] step 015/200  batch_loss=5.6671  full_loss=5.4254
[DPO] step 016/200  batch_los

In [84]:
import numpy as np

assert "pairs" in globals(), "pairs not loaded; run the cell that loads prefs_out/pairs.pkl."

pos_succ_flags = np.array([tr.success for (tr, _) in pairs], dtype=bool)
neg_succ_flags = np.array([tr.success for (_, tr) in pairs], dtype=bool)

print("Total pairs:", len(pairs))
print("Pos success rate:", pos_succ_flags.mean())
print("Neg success rate:", neg_succ_flags.mean())

mask_pos_better = np.logical_and(pos_succ_flags, ~neg_succ_flags)
mask_neg_better = np.logical_and(~pos_succ_flags, neg_succ_flags)
mask_both = np.logical_and(pos_succ_flags, neg_succ_flags)

print("Fraction pos_success=True & neg_success=False:", mask_pos_better.mean())
print("Fraction neg_success=True & pos_success=False:", mask_neg_better.mean())
print("Fraction both_success=True:", mask_both.mean())


Total pairs: 200
Pos success rate: 0.59
Neg success rate: 0.015
Fraction pos_success=True & neg_success=False: 0.575
Fraction neg_success=True & pos_success=False: 0.0
Fraction both_success=True: 0.015


In [ ]:
# Evaluation
import numpy as np
import pickle, os, torch
import torch.nn.functional as F
import robomimic.utils.torch_utils as TorchUtils
from robomimic.algo import RolloutPolicy

device = TorchUtils.get_torch_device(try_to_use_cuda=True)

if 'pairs' not in globals():
    with open("prefs_out/pairs.pkl", "rb") as f:
        pairs = pickle.load(f)

obs_dim = pairs[0][0].obs_flat[0].shape[0]
act_dim = pairs[0][0].actions[0].shape[0]

# Rotation helpers: 6D -> axis-angle, 10D -> 7D
def rot6d_to_axis_angle(rot6: np.ndarray, device=None) -> np.ndarray:
    """
    rot6: shape (6,)
    returns axis-angle: shape (3,)
    """
    import torch
    import torch.nn.functional as F

    if device is None:
        device = TorchUtils.get_torch_device(try_to_use_cuda=True)

    x = torch.as_tensor(rot6, dtype=torch.float32, device=device).view(1, 6)
    a1 = x[:, :3]
    a2 = x[:, 3:6]

    b1 = F.normalize(a1, dim=-1)
    a2_proj = (b1 * a2).sum(dim=-1, keepdim=True) * b1
    b2 = F.normalize(a2 - a2_proj, dim=-1)
    b3 = torch.cross(b1, b2, dim=-1)

    R = torch.stack([b1, b2, b3], dim=-1)  # (1,3,3)

    trace = R[:, 0, 0] + R[:, 1, 1] + R[:, 2, 2]
    cos = (trace - 1.0) / 2.0
    cos = torch.clamp(cos, -1.0 + 1e-7, 1.0 - 1e-7)
    angle = torch.acos(cos)  # (1,)

    sin = torch.sin(angle)
    axis = torch.zeros_like(b1)
    mask = sin.abs() > 1e-4
    if mask.any():
        num = torch.stack([
            R[:, 2, 1] - R[:, 1, 2],
            R[:, 0, 2] - R[:, 2, 0],
            R[:, 1, 0] - R[:, 0, 1],
        ], dim=-1)
        axis[mask] = F.normalize(num[mask] / (2.0 * sin[mask].unsqueeze(-1)), dim=-1)
    if (~mask).any():
        axis[~mask] = torch.tensor([1.0, 0.0, 0.0], device=device)

    axis_angle = axis * angle.unsqueeze(-1)
    return axis_angle.squeeze(0).detach().cpu().numpy()

def action_10d_to_7d(a10: np.ndarray, device=None) -> np.ndarray:
    """
    a10: [pos(3), rot6(6), grip(1)] -> [pos(3), axis_angle(3), grip(1)]
    """
    a10 = np.asarray(a10, dtype=np.float32)
    assert a10.shape[-1] == 10, f"Expected 10-D action, got {a10.shape}"
    pos = a10[:3]
    rot6 = a10[3:9]
    grip = a10[9:]
    aa = rot6d_to_axis_angle(rot6, device=device)
    return np.concatenate([pos, aa, grip], axis=-1)

# Post-processing: binary gripper at eval time
def postproc_action(a: np.ndarray, binary_gripper: bool = True, grip_idx: int = -1, clip: bool = True) -> np.ndarray:
    a = np.asarray(a, dtype=np.float32).copy()
    if binary_gripper and a.shape[0] >= 1:
        a[grip_idx] = 1.0 if a[grip_idx] >= 0.0 else -1.0
    if clip:
        np.clip(a, -1.0, 1.0, out=a)
    return a

def _succ(info: dict) -> bool:
    """
    Treat success as info["is_success"]["task"] == True if present.
    Fall back to other keys if needed.
    """
    if not isinstance(info, dict):
        return False

    # Main key we discovered:
    if "is_success" in info:
        succ = info["is_success"]
        # sometimes it's a dict like {"task": np.bool_}
        if isinstance(succ, dict) and "task" in succ:
            return bool(succ["task"])
        # sometimes it's a scalar bool / np.bool_
        try:
            return bool(succ)
        except Exception:
            pass

    return bool(
        info.get("success", False)
        or info.get("task_success", False)
        or info.get("goal_achieved", False)
    )

# Put BC nets into eval mode
model.nets.eval()

# base BC policy
policy_bc_eval = RolloutPolicy(model)

# BC-DPO policy (θ) as a BC algo
if 'policy_dpo_model' in globals():
    theta_model = policy_dpo_model
else:
    theta_model = get_example_model(dataset_path, device=device)
    theta_state = torch.load("prefs_out/bc_theta_dpo_nets.pt", map_location=device)
    for name, net in theta_model.nets.items():
        if name in theta_state:
            net.load_state_dict(theta_state[name])

# Put θ nets into eval mode
theta_model.nets.eval()

policy_dpo_eval = RolloutPolicy(theta_model)

# BC action helper for RolloutPolicy
def bc_act(policy_bc, obs: dict, deterministic: bool = False):
    """
    policy_bc: RolloutPolicy or BC algo-like object
    obs: obs dict from env
    returns: (action, None) where action is 1D np.array (10-D for lift)
    NOTE: ignores 'deterministic' flag and just calls policy_bc(obs).
    """
    out = policy_bc(obs)  # RolloutPolicy(obs_dict) -> action-like output

    if isinstance(out, dict):
        a = None
        for k in ("actions", "action", "raw_actions", "raw_action"):
            if k in out:
                a = out[k]
                break
        if a is None:
            raise RuntimeError("bc_act: dict output missing action key")
        a = np.asarray(a, dtype=np.float32).reshape(-1)
        return a, None

    if isinstance(out, (tuple, list)) and len(out) >= 1:
        a = np.asarray(out[0], dtype=np.float32).reshape(-1)
        return a, None

    a = np.asarray(out, dtype=np.float32).reshape(-1)
    return a, None

# eval helpers using 10D -> 7D conversion
def eval_policy(env, pol, n=50, horizon=200):
    returns, successes = [], 0
    for _ in range(n):
        obs = env.reset()
        if hasattr(pol, "reset"):
            try: pol.reset()
            except Exception: pass
        R = 0.0; last_info = {}
        for t in range(horizon):
            a10, _ = bc_act(pol, obs)   # 10-D
            a10 = postproc_action(a10, binary_gripper=True, grip_idx=-1, clip=True)
            a7 = action_10d_to_7d(a10, device=device)  # convert to 7-D for env
            obs, r, done, info = env.step(a7)
            R += float(r); last_info = info
            if done: break
        returns.append(R); successes += int(_succ(last_info))
    ret = np.asarray(returns, dtype=np.float32)
    return {
        "mean": float(ret.mean()),
        "std":  float(ret.std(ddof=0)),
        "min":  float(ret.min()),
        "max":  float(ret.max()),
        "successes": successes,
        "n":   n,
    }

H = 200

print("=== BC eval (with binary gripper) ===")
bc_stats_stoch = eval_policy(env, policy_bc_eval, n=20, horizon=H)
print(bc_stats_stoch)

print("\n=== DPO eval (with binary gripper) ===")
dpo_stats_greedy = eval_policy(env, policy_dpo_eval, n=20, horizon=H)
print(dpo_stats_greedy)



=== BC eval (with binary gripper) ===
ObservationKeyToModalityDict: robot0_joint_pos not found, adding robot0_joint_pos to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_joint_pos_cos not found, adding robot0_joint_pos_cos to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_joint_pos_sin not found, adding robot0_joint_pos_sin to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_joint_vel not found, adding robot0_joint_vel to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_eef_quat_site not found, adding robot0_eef_quat_site to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_gripper_qvel not found, adding robot0_gripper_qvel to mapping with assumed low_dim modality!
{'mean': 44.599998474121094, 'std': 68.83414459228516, 'min': 0.0, 'max': 161.0, 'successes': 5, 'n': 20}

=== DPO eval (with binary gripper) ===
{'mean': 142.60000610351562, 'std': 35.314

In [90]:
# Iteration 2: Collect preferences with the DPO policy
import os, pickle

from robomimic.algo import RolloutPolicy

# 1) Build a RolloutPolicy for the *DPO-trained* model
# If you still have theta_model / policy_dpo_model in memory:
collector_algo = policy_dpo_model          # this is your DPO θ from iter 1
# If not, load it from disk instead:
# collector_algo = get_example_model(dataset_path, device=device)
# theta_state = torch.load("prefs_out/bc_theta_dpo_nets.pt", map_location=device)
# for name, net in collector_algo.nets.items():
#     if name in theta_state:
#         net.load_state_dict(theta_state[name])

collector_algo.nets.eval()
policy_dpo_collect = RolloutPolicy(collector_algo)

N_PAIRS_2     = 200
HORIZON_2     = 200
TRIALS_PER_2  = 60

print("Collecting iteration-2 preference pairs with DPO policy (stochastic)...")
pairs_iter2 = collect_preference_pairs_minimal(
    env,
    policy_dpo_collect,
    n_pairs=N_PAIRS_2,
    horizon=HORIZON_2,
    min_return_gap=1e-6,
    prefer_higher=True,
    verbose_every=max(1, N_PAIRS_2 // 10),
)

print(f"[iter2] Collected {len(pairs_iter2)} pairs.")

os.makedirs("prefs_out", exist_ok=True)
with open("prefs_out/pairs_iter2.pkl", "wb") as f:
    pickle.dump(pairs_iter2, f)
print("[iter2] Saved raw preference pairs to prefs_out/pairs_iter2.pkl")


[collect-min] 20/200 pairs — pos_ret=90.0 (succ=True)  neg_ret=139.0 (succ=False)
[collect-min] 40/200 pairs — pos_ret=158.0 (succ=True)  neg_ret=156.0 (succ=True)
[collect-min] 60/200 pairs — pos_ret=157.0 (succ=True)  neg_ret=155.0 (succ=True)
[collect-min] 80/200 pairs — pos_ret=159.0 (succ=True)  neg_ret=158.0 (succ=True)
[collect-min] 100/200 pairs — pos_ret=158.0 (succ=True)  neg_ret=157.0 (succ=True)
[collect-min] 120/200 pairs — pos_ret=157.0 (succ=True)  neg_ret=156.0 (succ=True)
[collect-min] 140/200 pairs — pos_ret=125.0 (succ=True)  neg_ret=128.0 (succ=False)
[collect-min] 160/200 pairs — pos_ret=154.0 (succ=True)  neg_ret=153.0 (succ=True)
[collect-min] 180/200 pairs — pos_ret=158.0 (succ=True)  neg_ret=138.0 (succ=False)
[collect-min] 200/200 pairs — pos_ret=157.0 (succ=True)  neg_ret=156.0 (succ=True)
[iter2] Collected 200 pairs.
[iter2] Saved raw preference pairs to prefs_out/pairs_iter2.pkl


In [91]:
# Iteration 2: Build success-vs-failure pairs
import numpy as np, pickle, os

if "pairs_iter2" not in globals():
    with open("prefs_out/pairs_iter2.pkl", "rb") as f:
        pairs_iter2 = pickle.load(f)

pairs_sf_iter2 = []
for pos, neg in pairs_iter2:
    # enforce pos is success, neg is failure
    if getattr(pos, "success", False) and not getattr(neg, "success", False):
        pairs_sf_iter2.append((pos, neg))

print("[iter2] Total pairs:", len(pairs_iter2))
print("[iter2] Success-vs-failure pairs:", len(pairs_sf_iter2))

# pos_succ_2 = np.mean([p.success for p, _ in pairs_sf_iter2])
# neg_succ_2 = np.mean([n.success for _, n in pairs_sf_iter2])
# print("[iter2] pairs_sf_iter2 pos_success rate:", pos_succ_2)
# print("[iter2] pairs_sf_iter2 neg_success rate:", neg_succ_2)

# os.makedirs("prefs_out", exist_ok=True)
# with open("prefs_out/pairs_sf_iter2.pkl", "wb") as f:
#     pickle.dump(pairs_sf_iter2, f)
# print("[iter2] Saved success-vs-failure pairs to prefs_out/pairs_sf_iter2.pkl")


[iter2] Total pairs: 200
[iter2] Success-vs-failure pairs: 60


In [ ]:
# Iteration 2: DPO training on new preference dataset
import torch, pickle, numpy as np

device = TorchUtils.get_torch_device(try_to_use_cuda=True)

if "pairs_iter2" not in globals():
    with open("prefs_out/pairs_iter2.pkl", "rb") as f:
        pairs_sf_iter2 = pickle.load(f)

print("[iter2] Using", len(pairs_sf_iter2), "success-vs-failure pairs for DPO.")

bc_model = algo_to_device(model, device)   # BC algo; same as before
ref = BCTrajectoryReference(bc_model, device=device)

theta_model_iter2 = get_example_model(dataset_path, device=device)
theta_state_iter1 = torch.load("prefs_out/bc_theta_dpo_nets.pt", map_location=device)  # from iter1
for name, net in theta_model_iter2.nets.items():
    if name in theta_state_iter1:
        net.load_state_dict(theta_state_iter1[name])

theta_model_iter2.nets.to(device)
theta_model_iter2.nets.eval()

policy_theta_iter2 = BCTrajectoryThetaTorch(theta_model_iter2, device=device)

print("Pref-accuracy BEFORE DPO (iter2, sf):",
      pref_accuracy(policy_theta_iter2, ref, pairs_sf_iter2, beta=5e-7))

with torch.no_grad():
    vals_pi, vals_ref = [], []
    for i, (pos, neg) in enumerate(pairs_sf_iter2[:10]):
        for tr, tag in [(pos, "pos"), (neg, "neg")]:
            lp_pi  = policy_theta_iter2.logprob_traj_tensor(tr).item()
            lp_ref = ref.logprob_traj_tensor(tr).item()
            vals_pi.append(lp_pi)
            vals_ref.append(lp_ref)
            print(f"[iter2] pair {i} {tag}: pi={lp_pi:.2f}, ref={lp_ref:.2f}, diff={lp_pi - lp_ref:.2f}")
    print("[iter2] pi logp mean/std:", np.mean(vals_pi), np.std(vals_pi))
    print("[iter2] ref logp mean/std:", np.mean(vals_ref), np.std(vals_ref))

hist_iter2 = train_dpo(
    policy_theta_iter2,
    ref,
    pairs_sf_iter2,
    beta=5e-7,
    lr=1e-6,
    steps=200,
    batch_size=32,
    print_every=1,
    full_eval_every=5,
    bc_anchor_lambda=0.0,
)

print("Pref-accuracy AFTER  DPO (iter2, sf):",
      pref_accuracy(policy_theta_iter2, ref, pairs_sf_iter2, beta=5e-7))

os.makedirs("prefs_out", exist_ok=True)
theta_state_2 = {name: net.state_dict() for name, net in theta_model_iter2.nets.items()}
torch.save(theta_state_2, "prefs_out/bc_theta_dpo_nets_iter2.pt")
print("[iter2] Saved θ to prefs_out/bc_theta_dpo_nets_iter2.pt")

policy_dpo_model = theta_model_iter2


[iter2] Using 60 success-vs-failure pairs for DPO.

============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['robot0_eef_quat', 'object', 'robot0_gripper_qpos', 'robot0_eef_pos']
using obs modality: rgb with keys: []
using obs modality: depth with keys: []
using obs modality: scan with keys: []
Pref-accuracy BEFORE DPO (iter2, sf): 0.5333333611488342
[iter2] pair 0 pos: pi=-1840095.50, ref=-1856858.12, diff=16762.62
[iter2] pair 0 neg: pi=-2313090.00, ref=-2019091.88, diff=-293998.12
[iter2] pair 1 pos: pi=-1820657.62, ref=-2331793.50, diff=511135.88
[iter2] pair 1 neg: pi=-3068220.75, ref=-3272326.75, diff=204106.00
[iter2] pair 2 pos: pi=-1996482.50, ref=-1926997.12, diff=-69485.38
[iter2] pair 2 neg: pi=-1197854.12, ref=-1214507.38, diff=16653.25
[iter2] pair 3 pos: pi=-2050369.75, ref=-2689339.50, diff=638969.75
[iter2] pair 3 neg: pi=-2809600.75, ref=-2720551.00, diff=-89049.75
[iter2] pair 4 pos: pi=-1472003.25, ref=-1395785

In [93]:
print("=== BC eval (with binary gripper) ===")
bc_stats_stoch = eval_policy(env, policy_bc_eval, n=20, horizon=200)
print(bc_stats_stoch)

print("\n=== DPO iter2 eval (with binary gripper) ===")
dpo_stats_greedy = eval_policy(env, policy_dpo_eval, n=20, horizon=200)
print(dpo_stats_greedy)


=== BC eval (with binary gripper) ===
ObservationKeyToModalityDict: robot0_joint_pos not found, adding robot0_joint_pos to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_joint_pos_cos not found, adding robot0_joint_pos_cos to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_joint_pos_sin not found, adding robot0_joint_pos_sin to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_joint_vel not found, adding robot0_joint_vel to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_eef_quat_site not found, adding robot0_eef_quat_site to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_gripper_qvel not found, adding robot0_gripper_qvel to mapping with assumed low_dim modality!
{'mean': 55.099998474121094, 'std': 68.62353515625, 'min': 0.0, 'max': 154.0, 'successes': 3, 'n': 20}

=== DPO iter2 eval (with binary gripper) ===
{'mean': 127.05000305175781, 'std': 54.

## What's next?

Now that you understand the basic components of robomimic, it's time to delve deeper into each component by reading up the [documentation site](https://robomimic.github.io/docs/introduction/overview.html). Robomimic offers a rich set of utilities for model building, training loop management, model checkpointing, visualization, hyper-parameter sweeping, and logging. You can get to more about each component by following the provided the examples and tutorials.

